In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
games_ds = load_dataset("davdsdfd/nba-games","games")
teams_ds = load_dataset("davdsdfd/nba-games","teams")

README.md:   0%|          | 0.00/925 [00:00<?, ?B/s]

games.csv:   0%|          | 0.00/4.50M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29755 [00:00<?, ? examples/s]

teams.csv:   0%|          | 0.00/4.06k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

In [4]:
games_df = games_ds["train"].to_pandas()
teams_df = teams_ds["train"].to_pandas()

In [5]:
team_names = teams_df[["TEAM_ID", "NICKNAME"]]

games_df = (
    games_df
    .merge(
        team_names,
        left_on="HOME_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    )
    .rename(columns={"NICKNAME": "HOME_TEAM"})
    .drop(columns="TEAM_ID")
    .merge(
        team_names,
        left_on="VISITOR_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    )
    .rename(columns={"NICKNAME": "VISITOR_TEAM"})
    .drop(columns="TEAM_ID")
)

In [6]:
games_df["GAME_DATE_EST"] = pd.to_datetime(games_df["GAME_DATE_EST"])
games_df = games_df.sort_values("GAME_DATE_EST").reset_index(drop=True)

base_games = games_df[[
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "HOME_TEAM",
    "VISITOR_TEAM",
    "PTS_home",
    "PTS_away",
    "FG_PCT_home",
    "FG_PCT_away",
    "FT_PCT_home",
    "FT_PCT_away",
    "FG3_PCT_home",
    "FG3_PCT_away",
    "AST_home",
    "AST_away",
    "REB_home",
    "REB_away",
    "HOME_TEAM_WINS"
]].copy()

home_games = base_games.copy()
away_games = base_games.copy()

home_games = home_games.rename(columns={
    "HOME_TEAM": "TEAM",
    "VISITOR_TEAM": "OPPONENT",
    "PTS_home": "PTS",
    "PTS_away": "PTS_ALLOWED",
    "FG_PCT_home": "FG_PCT",
    "FT_PCT_home": "FT_PCT",
    "FG3_PCT_home": "FG3_PCT",
    "AST_home": "AST",
    "REB_home": "REB",
    "HOME_TEAM_WINS": "WIN"
})

home_games["IS_HOME"] = 1

away_games = away_games.rename(columns={
    "VISITOR_TEAM": "TEAM",
    "HOME_TEAM": "OPPONENT",
    "PTS_away": "PTS",
    "PTS_home": "PTS_ALLOWED",
    "FG_PCT_away": "FG_PCT",
    "FT_PCT_away": "FT_PCT",
    "FG3_PCT_away": "FG3_PCT",
    "AST_away": "AST",
    "REB_away": "REB"
})

away_games["WIN"] = 1 - away_games["HOME_TEAM_WINS"]
away_games["IS_HOME"] = 0

In [7]:
team_games_df = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

team_games_df = team_games_df.sort_values(
    ["TEAM", "GAME_DATE_EST", "GAME_ID"]
).reset_index(drop=True)

In [8]:
group = team_games_df.groupby(["TEAM", "SEASON"])

team_games_df["GAMES_BEFORE"] = group.cumcount()

team_games_df["WINS_BEFORE"] = group["WIN"].transform(
    lambda x: x.shift(1).cumsum()
)

team_games_df["WIN_PCT_BEFORE"] = (
    team_games_df["WINS_BEFORE"] /
    team_games_df["GAMES_BEFORE"]
)

stats = [
    "PTS",
    "PTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for stat in stats:
    team_games_df[f"{stat}_LAST_10"] = (
        group[stat]
        .transform(
            lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        )
    )

In [9]:
stats = [
    "PTS",
    "PTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for stat in stats:
    team_games_df[f"{stat}_LAST_10"] = (
        team_games_df
        .groupby(["TEAM", "SEASON"])[stat]
        .transform(
            lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        )
    )

In [10]:
feature_cols = [
    "WIN_PCT_BEFORE",
    "PTS_LAST_10",
    "PTS_ALLOWED_LAST_10",
    "FG_PCT_LAST_10",
    "FT_PCT_LAST_10",
    "FG3_PCT_LAST_10",
    "AST_LAST_10",
    "REB_LAST_10"
]

home_features = team_games_df[
    team_games_df["IS_HOME"] == 1
][["GAME_ID"] + feature_cols].copy()

away_features = team_games_df[
    team_games_df["IS_HOME"] == 0
][["GAME_ID"] + feature_cols].copy()

home_features = home_features.rename(
    columns={
        col: f"HOME_{col}"
        for col in feature_cols
    }
)

away_features = away_features.rename(
    columns={
        col: f"AWAY_{col}"
        for col in feature_cols
    }
)

In [11]:
model_df = home_features.merge(
    away_features,
    on="GAME_ID",
    how="inner"
)

In [12]:
game_info = games_df[[
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "HOME_TEAM",
    "VISITOR_TEAM",
    "HOME_TEAM_WINS"
]].copy()

In [13]:
model_df = game_info.merge(
    model_df,
    on="GAME_ID",
    how="inner"
)

In [14]:
model_df["WIN_PCT_DIFF"] = (
    model_df["HOME_WIN_PCT_BEFORE"]
    - model_df["AWAY_WIN_PCT_BEFORE"]
)

for stat in [
    "PTS_LAST_10",
    "PTS_ALLOWED_LAST_10",
    "FG_PCT_LAST_10",
    "FT_PCT_LAST_10",
    "FG3_PCT_LAST_10",
    "AST_LAST_10",
    "REB_LAST_10"
]:
    model_df[f"{stat}_DIFF"] = (
        model_df[f"HOME_{stat}"]
        - model_df[f"AWAY_{stat}"]
    )

In [15]:
model_df.tail()

,GAME_ID,GAME_DATE_EST,SEASON,HOME_TEAM,VISITOR_TEAM,HOME_TEAM_WINS,HOME_WIN_PCT_BEFORE,HOME_PTS_LAST_10,HOME_PTS_ALLOWED_LAST_10,HOME_FG_PCT_LAST_10,...,AWAY_AST_LAST_10,AWAY_REB_LAST_10,WIN_PCT_DIFF,PTS_LAST_10_DIFF,PTS_ALLOWED_LAST_10_DIFF,FG_PCT_LAST_10_DIFF,FT_PCT_LAST_10_DIFF,FG3_PCT_LAST_10_DIFF,AST_LAST_10_DIFF,REB_LAST_10_DIFF
29924,22400774,2025-02-12,2024,Raptors,Cavaliers,0,0.338983,108.1,108.7,0.4497,...,29.3,46.9,-0.415403,-18.1,-5.1,-0.0551,-0.0115,-0.0431,-1.4,-3.4
29925,22400773,2025-02-12,2024,Knicks,Hawks,1,0.672414,121.5,115.0,0.4940,...,28.4,42.8,0.206897,8.2,-1.9,0.0278,0.0593,0.0076,1.7,-0.8
29926,22400772,2025-02-12,2024,Nets,76ers,1,0.350877,98.6,99.5,0.4183,...,25.7,39.0,-0.028433,-16.3,-15.5,-0.0639,-0.0046,-0.0685,-3.4,4.9
29927,22400770,2025-02-12,2024,Magic,Hornets,1,0.465517,104.5,112.1,0.4389,...,24.4,44.4,0.197660,0.0,3.3,0.0053,-0.0712,-0.0361,-2.5,-2.9
29928,22400769,2025-02-12,2024,Celtics,Spurs,1,0.711864,115.4,108.5,0.4696,...,30.3,43.6,0.265436,-4.0,-11.5,-0.0167,0.0265,0.0098,-2.7,0.9


In [16]:
features = [
    "WIN_PCT_DIFF",
    "PTS_LAST_10_DIFF",
    "PTS_ALLOWED_LAST_10_DIFF",
    "FG_PCT_LAST_10_DIFF",
    "FT_PCT_LAST_10_DIFF",
    "FG3_PCT_LAST_10_DIFF",
    "AST_LAST_10_DIFF",
    "REB_LAST_10_DIFF"
]

target = "HOME_TEAM_WINS"

model_df = model_df.dropna(
    subset=features + [target]
).reset_index(drop=True)

X = model_df[features]
y = model_df[target]